In [2]:
import os
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
dataset_path = "/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/raw/UAH-DRIVESET-v1"

print(dataset_path)

/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/raw/UAH-DRIVESET-v1


In [4]:
sample_driver = "D1"

sample_trip = "20151110175712-16km-D1-NORMAL1-SECONDARY"

trip_path = os.path.join(
    dataset_path,
    sample_driver,
    sample_trip
)

print(trip_path)

/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/raw/UAH-DRIVESET-v1/D1/20151110175712-16km-D1-NORMAL1-SECONDARY


In [5]:
vehicle_path = os.path.join(
    trip_path,
    "PROC_VEHICLE_DETECTION.txt"
)

with open(vehicle_path, "r") as f:

    for i in range(20):

        print(f.readline().rstrip())

7.11 -1.00 -1.00 0 67.4
7.22 17.97 0.99 2 65.2
7.31 17.97 0.99 2 65.2
7.43 18.04 1.00 2 65.2
7.52 18.12 1.00 2 65.2
7.62 18.10 1.00 2 65.2
7.73 18.07 1.00 2 65.2
7.85 18.16 1.00 2 65.2
7.99 18.12 1.00 2 65.2
8.10 17.69 0.99 2 64.5
8.21 17.74 0.99 2 64.5
8.32 17.78 0.99 2 64.5
8.44 17.94 1.00 2 64.5
8.55 18.07 1.01 2 64.5
8.66 18.17 1.01 3 64.5
8.77 18.13 1.01 3 64.5
8.90 18.10 1.01 3 64.5
9.03 17.83 1.01 3 63.6
9.13 17.62 1.00 3 63.6
9.25 16.68 0.94 4 63.6


In [6]:
vehicle_df = pd.read_csv(
    vehicle_path,
    sep=r"\s+",
    header=None
)

vehicle_df.head()

,0,1,2,3,4
0,7.11,-1.00,-1.00,0,67.4
1,7.22,17.97,0.99,2,65.2
2,7.31,17.97,0.99,2,65.2
3,7.43,18.04,1.00,2,65.2
4,7.52,18.12,1.00,2,65.2


In [7]:
print(vehicle_df.shape)

print()

vehicle_df.info()

print()

print(vehicle_df.describe())

(4739, 5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4739 entries, 0 to 4738
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       4739 non-null   float64
 1   1       4739 non-null   float64
 2   2       4739 non-null   float64
 3   3       4739 non-null   int64  
 4   4       4739 non-null   float64
dtypes: float64(4), int64(1)
memory usage: 185.2 KB

                 0            1            2            3            4
count  4739.000000  4739.000000  4739.000000  4739.000000  4739.000000
mean    313.376024    16.558983     0.311165     0.771893    96.099472
std     184.958954    20.803357     1.121112     0.555569     9.805397
min       7.110000    -1.000000    -1.000000     0.000000    60.900000
25%     146.315000    -1.000000    -1.000000     0.000000    89.700000
50%     317.310000    16.840000     0.650000     1.000000    95.500000
75%     474.620000    23.140000     0.930000     1.000000   104.300000
max

In [8]:
vehicle_df.columns = [

    "time",

    "front_distance",

    "relative_speed",

    "vehicle_state",

    "confidence"

]

vehicle_df.head()

,time,front_distance,relative_speed,vehicle_state,confidence
0,7.11,-1.00,-1.00,0,67.4
1,7.22,17.97,0.99,2,65.2
2,7.31,17.97,0.99,2,65.2
3,7.43,18.04,1.00,2,65.2
4,7.52,18.12,1.00,2,65.2


In [9]:
print(vehicle_df.isnull().sum())

print()

print(vehicle_df["vehicle_state"].value_counts())

print()

print(vehicle_df["front_distance"].describe())

print()

print(vehicle_df["relative_speed"].describe())

time              0
front_distance    0
relative_speed    0
vehicle_state     0
confidence        0
dtype: int64

vehicle_state
1    3088
0    1375
2     260
3      14
4       2
Name: count, dtype: int64

count    4739.000000
mean       16.558983
std        20.803357
min        -1.000000
25%        -1.000000
50%        16.840000
75%        23.140000
max       200.020000
Name: front_distance, dtype: float64

count    4739.000000
mean        0.311165
std         1.121112
min        -1.000000
25%        -1.000000
50%         0.650000
75%         0.930000
max         7.220000
Name: relative_speed, dtype: float64


In [10]:
def load_vehicle_detection(vehicle_path):

    vehicle_df = pd.read_csv(

        vehicle_path,

        sep=r"\s+",

        header=None

    )

    vehicle_df.columns = [

        "time",

        "front_distance",

        "relative_speed",

        "vehicle_state",

        "confidence"

    ]

    return vehicle_df

In [11]:
def clean_vehicle_detection(vehicle_df):

    vehicle_df = vehicle_df.copy()

    vehicle_df = vehicle_df.sort_values(

        "time"

    )

    vehicle_df = vehicle_df.reset_index(

        drop=True

    )

    return vehicle_df

In [12]:
def merge_vehicle_data(master_df, vehicle_df):

    master_df = master_df.copy()
    vehicle_df = vehicle_df.copy()

    master_df = master_df.sort_values(
        "timestamp"
    )

    vehicle_df = vehicle_df.sort_values(
        "time"
    )

    merged_df = pd.merge_asof(

        master_df,

        vehicle_df,

        left_on="timestamp",

        right_on="time",

        direction="nearest"

    )

    return merged_df

In [13]:
vehicle_df = load_vehicle_detection(
    vehicle_path
)

vehicle_df = clean_vehicle_detection(
    vehicle_df
)

vehicle_df.head()

,time,front_distance,relative_speed,vehicle_state,confidence
0,7.11,-1.00,-1.00,0,67.4
1,7.22,17.97,0.99,2,65.2
2,7.31,17.97,0.99,2,65.2
3,7.43,18.04,1.00,2,65.2
4,7.52,18.12,1.00,2,65.2
